## Import libraries

In [1]:
import csv
import numpy as np
import pandas as pd
import config

In [2]:
# Define Paths
YAHOO_RAW_DIR = config.RAW_DATA_DIR / "yahoo"
FRED_RAW_DIR = config.RAW_DATA_DIR / "fred"
FAMA_FRENCH_RAW_DIR = config.RAW_DATA_DIR / "fama_french"

YAHOO_INPUT_DIR = config.INPUT_DATA_DIR / "yahoo"
FRED_INPUT_DIR = config.INPUT_DATA_DIR / "fred"
FAMA_FRENCH_INPUT_DIR = config.INPUT_DATA_DIR / "fama_french"

YAHOO_INPUT_DIR.mkdir(parents=True, exist_ok=True)
FRED_INPUT_DIR.mkdir(parents=True, exist_ok=True)
FAMA_FRENCH_INPUT_DIR.mkdir(parents=True, exist_ok=True)

# Read the master investable universe
universe = pd.read_csv(config.UNIVERSE_FILE)

## Functions

In [3]:
def clean_yahoo_file(input_file, output_file):

    # Read the raw file
    data = pd.read_csv(input_file)

    # Convert Date
    data["Date"] = pd.to_datetime(data["Date"])

    # Get the Yahoo ticker from the filename
    ticker = input_file.stem

    # Check and remove rows with missing values
    missing_rows = data.isna().any(axis=1).sum()

    if missing_rows > 0:
        data = data.dropna()
        print(f"{ticker}: {missing_rows} rows with missing values were found and removed.")

    # Check and remove duplicate dates
    duplicate_rows = data["Date"].duplicated(keep="last").sum()

    if duplicate_rows > 0:
        data = data.drop_duplicates(subset="Date", keep="last")
        print(f"{ticker}: {duplicate_rows} duplicate dates were found and removed.")

    # Sort chronologically
    data = data.sort_values("Date")

    # Save cleaned file
    data.to_csv(output_file, index=False)

    return data


def clean_fred_file(input_file, output_file):

    # Read the raw file
    data = pd.read_csv(input_file)

    # Convert Date
    data["Date"] = pd.to_datetime(data["Date"])

    # Get the FRED series name from the filename
    series_name = input_file.stem

     # Check and remove rows with missing values
    missing_rows = data.isna().any(axis=1).sum()

    if missing_rows > 0:
        data = data.dropna()
        print(f"{series_name}: {missing_rows} rows with missing values were found and removed.")

    # Check and remove duplicate dates
    duplicate_rows = data["Date"].duplicated(keep="last").sum()

    if duplicate_rows > 0:
        data = data.drop_duplicates(subset="Date", keep="last")
        print(f"{series_name}: {duplicate_rows} duplicate dates were found and removed.")

    # Sort chronologically
    data = data.sort_values("Date")

    # Save cleaned file
    data.to_csv(output_file, index=False)

    return data


def clean_fama_french_file(input_file, output_file):
    """
    Clean a Kenneth French CSV file by:
    - removing metadata rows before the actual header
    - using the existing column names
    - adding 'Date' as the first column name
    - removing non-data rows after the observations
    - converting Date to datetime
    """

    # Read the raw file line by line
    with open(input_file, "r", encoding="latin1") as f:
        rows = list(csv.reader(f))

    # Find the first row where the first cell is a YYYYMMDD date
    data_row = None

    for i, row in enumerate(rows):
        if len(row) > 0:
            first_cell = row[0].strip()

            if first_cell.isdigit() and len(first_cell) == 8 and first_cell.startswith("19"):
                data_row = i
                break

    if data_row is None:
        raise ValueError(f"Could not identify the first data row in {input_file}")

    # The row immediately above contains the header
    header_row = data_row - 1

    # Get the existing column names from the header
    columns = rows[header_row]

    # Rename only the first column
    columns[0] = "Date"

    # Keep rows from the first data row onward
    data_rows = rows[data_row:]

    # Keep only rows whose first cell is a valid YYYYMMDD date
    data_rows = [
        row for row in data_rows
        if len(row) > 0
        and row[0].strip().isdigit()
        and len(row[0].strip()) == 8
    ]

    # Create DataFrame
    data = pd.DataFrame(data_rows, columns=columns)

    # Convert Date to datetime
    data["Date"] = pd.to_datetime(data["Date"], format="%Y%m%d")

    # Keep observations up to and including AS_OF_DATE
    data = data.loc[data["Date"] <= pd.Timestamp(config.AS_OF_DATE)].copy()

    # Get the file name
    file_name = input_file.stem

    # Check and remove rows with missing values
    missing_rows = data.isna().any(axis=1).sum()

    if missing_rows > 0:
        data = data.dropna()
        print(f"{file_name}: {missing_rows} rows with missing values were found and removed.")

    # Check and remove duplicate dates
    duplicate_rows = data["Date"].duplicated(keep="last").sum()

    if duplicate_rows > 0:
        data = data.drop_duplicates(subset="Date", keep="last")
        print(f"{file_name}: {duplicate_rows} duplicate dates were found and removed.")

    # Sort chronologically
    data = data.sort_values("Date")

    # Save cleaned file
    data.to_csv(output_file, index=False)

    return data


def calculate_corwin_schultz_spread(data, SPREAD_WINDOW, SPREAD_MIN_OBS):
    """
    Calculate the Corwin-Schultz proportional full bid-ask spread.

    The raw estimator uses two adjacent trading days.
    Negative estimates are set to zero.

    The final Spread column is the trailing 20-trading-day average
    of those estimates, requiring at least 12 valid estimates.
    """

    high = pd.to_numeric(data["High"], errors="coerce")
    low = pd.to_numeric(data["Low"], errors="coerce")
    close = pd.to_numeric(data["Close"], errors="coerce")
    volume = pd.to_numeric(data["Volume"], errors="coerce")

    valid = ((high > low) & (low > 0) & (close > 0) & (volume > 0))

    high = high.where(valid)
    low = low.where(valid)
    close = close.where(close > 0)

    # Values from the previous trading day.
    prev_high = high.shift(1)
    prev_low = low.shift(1)
    prev_close = close.shift(1)

    adjusted_high = high.copy()
    adjusted_low = low.copy()

    # -------------------------------------------------------------------------
    # Remove upward overnight gaps
    # -------------------------------------------------------------------------

    gap_up = ((prev_close > 0) & (prev_close < low))
    adjusted_high = adjusted_high.where(~gap_up, high - (low - prev_close))
    adjusted_low = adjusted_low.where(~gap_up, prev_close)

    # -------------------------------------------------------------------------
    # Remove downward overnight gaps
    # -------------------------------------------------------------------------

    gap_down = ((prev_close > 0) & (prev_close > high))
    adjusted_high = adjusted_high.where(~gap_down, prev_close)
    adjusted_low = adjusted_low.where(~gap_down, low + (prev_close - high))

    valid_pair = (
        (adjusted_high > adjusted_low)
        & (adjusted_low > 0)
        & (prev_high > prev_low)
        & (prev_low > 0)
    )

    adjusted_high = adjusted_high.where(valid_pair)
    adjusted_low = adjusted_low.where(valid_pair)

    # -------------------------------------------------------------------------
    # Corwin-Schultz beta
    # -------------------------------------------------------------------------

    beta = (np.log(adjusted_high / adjusted_low).pow(2) + np.log(prev_high / prev_low).pow(2))

    # -------------------------------------------------------------------------
    # Corwin-Schultz gamma
    # -------------------------------------------------------------------------

    two_day_high = pd.concat([adjusted_high, prev_high], axis=1).max(axis=1, skipna=False)
    two_day_low = pd.concat([adjusted_low, prev_low], axis=1).min(axis=1, skipna=False)
    gamma = np.log(two_day_high / two_day_low).pow(2)

    # -------------------------------------------------------------------------
    # Alpha and spread
    # -------------------------------------------------------------------------

    constant = (3 - 2 * np.sqrt(2))
    alpha = ((np.sqrt(2 * beta) - np.sqrt(beta)) / constant - np.sqrt(gamma / constant))
    raw_spread = (2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha))).clip(lower=0)

    # Smooth the noisy two-day estimates.
    spread = raw_spread.rolling(window=SPREAD_WINDOW, min_periods=SPREAD_MIN_OBS).mean()

    return spread


def calculate_adv_dollars(data, ADV_WINDOW):
    """
    Calculate six-month Average Daily Dollar Volume.

    Dollar Volume_t: Close_t * Volume_t
    ADV_t: average Dollar Volume over the previous 126 trading days, including date t.
    """

    close = pd.to_numeric(data["Close"], errors="coerce")
    volume = pd.to_numeric(data["Volume"], errors="coerce")
    dollar_volume = (close * volume).where((close > 0) & (volume > 0))
    adv_dollars = dollar_volume.rolling(window=ADV_WINDOW, min_periods=ADV_WINDOW).mean()

    return adv_dollars


def add_liquidity_columns(file_path, asset_class, SPREAD_WINDOW, SPREAD_MIN_OBS, ADV_WINDOW):
    """
    Add Spread and ADV_Dollars to one cleaned Yahoo OHLCV file.
    The existing CSV is overwritten with the two additional columns.
    """

    data = pd.read_csv(file_path)
    data["Date"] = pd.to_datetime(data["Date"])
    data = data.sort_values("Date").drop_duplicates("Date", keep="last").reset_index(drop=True)

    required_columns = {"High", "Low", "Close", "Volume"}
    missing_columns = (required_columns - set(data.columns))

    if missing_columns:
        raise ValueError(
            f"{file_path.stem}: missing columns "
            f"{sorted(missing_columns)}"
        )

    # We are deliberately not applying the equity/ETF microstructure model to Bitcoin
    if (str(asset_class).strip().lower() == "crypto"):
        data["Spread"] = np.nan
        data["ADV_Dollars"] = np.nan

    else:
        data["Spread"] = (calculate_corwin_schultz_spread(data, SPREAD_WINDOW, SPREAD_MIN_OBS))
        data["ADV_Dollars"] = (calculate_adv_dollars(data, ADV_WINDOW))

    data.to_csv(file_path, index=False)

    print(
        f"{file_path.stem}: "
        f"{data['Spread'].notna().sum():,} spread values, "
        f"{data['ADV_Dollars'].notna().sum():,} ADV values"
    )


def audit_yahoo_data(input_dir, output_file):

    audit_records = []

    for input_file in input_dir.glob("*.csv"):

        data = pd.read_csv(input_file)
        data["Date"] = pd.to_datetime(data["Date"])
        ticker = input_file.stem

        audit_records.append({
            "Ticker": ticker,
            "Start Date": data["Date"].min().date(),
            "End Date": data["Date"].max().date(),
            "Observations": len(data),
            "Missing OHLC": data[["Open", "High", "Low", "Close"]].isna().any(axis=1).sum(),
            "Missing Adj Close": data["Adj Close"].isna().sum(),
            "Missing Volume": data["Volume"].isna().sum(),
            "Duplicate Dates": data["Date"].duplicated().sum()
        })

    audit_data = pd.DataFrame(audit_records)
    audit_data.to_csv(output_file, index=False)

    return audit_data


def audit_fred_data(input_dir, output_file):

    audit_records = []

    for input_file in input_dir.glob("*.csv"):

        data = pd.read_csv(input_file)
        data["Date"] = pd.to_datetime(data["Date"])
        series_name = input_file.stem

        audit_records.append({
            "Series": series_name,
            "Start Date": data["Date"].min().date(),
            "End Date": data["Date"].max().date(),
            "Observations": len(data),
            "Missing Values": data.isna().any(axis=1).sum(),
            "Duplicate Dates": data["Date"].duplicated().sum()
        })

    audit_data = pd.DataFrame(audit_records)
    audit_data.to_csv(output_file, index=False)
    
    return audit_data


def audit_fama_french_data(input_dir, output_file):

    audit_records = []

    for input_file in input_dir.glob("*.csv"):

        data = pd.read_csv(input_file)
        data["Date"] = pd.to_datetime(data["Date"])

        file_name = input_file.stem

        audit_records.append({
            "File": file_name,
            "Start Date": data["Date"].min().date(),
            "End Date": data["Date"].max().date(),
            "Observations": len(data),
            "Missing Values": data.isna().any(axis=1).sum(),
            "Duplicate Dates": data["Date"].duplicated().sum()
        })

    audit_data = pd.DataFrame(audit_records)

    audit_data.to_csv(output_file, index=False)

    return audit_data

## Yahoo Finance Files

In [4]:
for input_file in YAHOO_RAW_DIR.glob("*.csv"):
    clean_yahoo_file(
        input_file=input_file,
        output_file=YAHOO_INPUT_DIR / input_file.name
    )

print("\nYahoo Finance data cleaning is completed.")


Yahoo Finance data cleaning is completed.


## FRED Files

In [5]:
for input_file in FRED_RAW_DIR.glob("*.csv"):
    clean_fred_file(
        input_file=input_file,
        output_file=FRED_INPUT_DIR / input_file.name
    )

print("\nFRED data cleaning is completed.")

AAA10Y: 480 rows with missing values were found and removed.
BAA10Y: 444 rows with missing values were found and removed.
CPIAUCSL: 1 rows with missing values were found and removed.
DCOILWTICO: 374 rows with missing values were found and removed.
DGS10: 721 rows with missing values were found and removed.
DGS2: 553 rows with missing values were found and removed.
DGS30: 546 rows with missing values were found and removed.
DGS3MO: 493 rows with missing values were found and removed.
DTWEXBGS: 211 rows with missing values were found and removed.
T10Y2Y: 553 rows with missing values were found and removed.
T10Y3M: 485 rows with missing values were found and removed.
T10YIE: 255 rows with missing values were found and removed.
T5YIE: 255 rows with missing values were found and removed.
T5YIFR: 255 rows with missing values were found and removed.
THREEFYTP10: 415 rows with missing values were found and removed.
THREEFYTP2: 415 rows with missing values were found and removed.
THREEFYTP5: 41

## Fama-French Files

In [6]:
# 5-factor file
ff5_data = clean_fama_french_file(
    input_file=FAMA_FRENCH_RAW_DIR / "F-F_Research_Data_5_Factors_2x3_daily.csv",
    output_file=FAMA_FRENCH_INPUT_DIR / "Fama_French_5_Factors_Daily.csv"
)

# Momentum file
mom_data = clean_fama_french_file(
    input_file=FAMA_FRENCH_RAW_DIR / "F-F_Momentum_Factor_daily.csv",
    output_file=FAMA_FRENCH_INPUT_DIR / "Momentum_Factor_Daily.csv"
)

print("\nFama-French data cleaning is completed.")


Fama-French data cleaning is completed.


## Liquidity Data

In [7]:
yahoo_assets = universe[
        universe["Source"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("yahoo finance")
    ][
        ["Ticker", "Asset Class"]
    ].drop_duplicates("Ticker")

for _, asset in yahoo_assets.iterrows():

    ticker = asset["Ticker"]
    file_path = YAHOO_INPUT_DIR / f"{ticker}.csv"

    if not file_path.exists():
        raise FileNotFoundError(f"Missing Yahoo file: {file_path}")

    add_liquidity_columns(
        file_path=file_path,
        asset_class=asset["Asset Class"],
        SPREAD_WINDOW=config.SPREAD_WINDOW,
        SPREAD_MIN_OBS=config.SPREAD_MIN_OBS,
        ADV_WINDOW=config.ADV_WINDOW
    )

print("\nSpread and ADV_Dollars added to all Yahoo OHLCV files.")

SPY: 8,492 spread values, 8,379 ADV values
VT: 4,611 spread values, 4,498 ADV values
QQQ: 6,950 spread values, 6,837 ADV values
VTV: 5,720 spread values, 5,607 ADV values
VUG: 5,718 spread values, 5,569 ADV values
IWM: 6,642 spread values, 6,529 ADV values
DVY: 5,776 spread values, 5,663 ADV values
VEA: 4,843 spread values, 4,730 ADV values
VGK: 5,441 spread values, 5,328 ADV values
FEZ: 6,039 spread values, 5,874 ADV values
EWJ: 7,698 spread values, 7,588 ADV values
FXI: 5,546 spread values, 5,433 ADV values
AFK: 4,585 spread values, 4,229 ADV values
VWO: 5,441 spread values, 5,328 ADV values
IPO: 3,261 spread values, 2,683 ADV values
IXP: 5,898 spread values, 5,538 ADV values
SMH: 6,637 spread values, 6,524 ADV values
IXG: 5,894 spread values, 5,534 ADV values
IXN: 5,951 spread values, 5,607 ADV values
XBI: 5,212 spread values, 5,099 ADV values
IXJ: 6,224 spread values, 5,904 ADV values
PBW: 5,446 spread values, 5,333 ADV values
VNQ: 5,553 spread values, 5,440 ADV values
RWX: 4,992 s

## Data Audit

In [8]:
# Yahoo
yahoo_audit = audit_yahoo_data(
    input_dir=YAHOO_INPUT_DIR,
    output_file=config.INPUT_DATA_DIR / "yahoo_data_audit.csv"
)

# FRED
fred_audit = audit_fred_data(
    input_dir=FRED_INPUT_DIR,
    output_file=config.INPUT_DATA_DIR / "fred_data_audit.csv"
)

# Fama-French
fama_french_audit = audit_fama_french_data(
    input_dir=FAMA_FRENCH_INPUT_DIR,
    output_file=config.INPUT_DATA_DIR / "fama_french_data_audit.csv"
)